In [1]:
import sys
import os
import json
import h5py

sys.path.append('D:\\Gdrive\\__CoChem\\GitHub-Repo\\CoChem-SCRIBE')

print('Starting Audit 070: OpenAlex LLM Provenance')

try:
    from cochem_scribe.core.metadata import SCRIBEMetadataEngine
except ImportError as e:
    print(f'FAIL: Could not import metadata engine: {e}')

from scribe_inference import ScribeInferenceEngine

if os.path.exists('cochem_state.h5'):
    os.remove('cochem_state.h5')
if os.path.exists('cochem_spycfit_state.json'):
    os.remove('cochem_spycfit_state.json')

engine = ScribeInferenceEngine()
engine.model_name = 'gpt-4-turbo'
with open('scribe_prompt_payload.txt', 'w') as f:
    f.write('Generate the introduction section of a report.')

engine.generate_document()

prov = {}
if os.path.exists('cochem_spycfit_state.json'):
    with open('cochem_spycfit_state.json', 'r') as f:
        jdata = json.load(f)
        prov.update(jdata.get('provenance_algorithms', {}))

if os.path.exists('cochem_state.h5'):
    with h5py.File('cochem_state.h5', 'r') as f:
        if 'provenance_algorithms' in f.attrs:
            prov.update(json.loads(f.attrs['provenance_algorithms']))

print('Intercepted provenance_algorithms:', prov)

if not prov:
    print('FAIL: System output an empty citation block for the primary text generation engine.')
elif 'gpt-4' not in str(prov).lower() and 'arxiv:2303.08774' not in str(prov).lower():
    print('FAIL: GPT-4 DOI not found in provenance.')
else:
    print('PASS')


Starting Audit 070: OpenAlex LLM Provenance
FAIL: Could not import metadata engine: No module named 'cochem_scribe.core.metadata'
Intercepted provenance_algorithms: {}
FAIL: System output an empty citation block for the primary text generation engine.
